In [ ]:
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive')
!gcloud config set project YOUR_GCP_PROJECT_ID
%cd /content/drive/MyDrive/E231project
!ls benchmarks infra scripts config.yaml
!grep -c "group_normal_eq" federated/local_als.py      # > 0 = vectorised federated code
!grep -c "_release_round_garbage" federated/fedavg_coordinator.py   # 2 = per-round cleanup

In [ ]:
# Inputs from earlier sessions must exist: canonical training data and the sweep subsets
!gcloud storage ls gs://YOUR_BUCKET/full/processed/
!gcloud storage ls gs://YOUR_BUCKET/full/scaling_scratch/subsets/

In [ ]:
# OPTIONAL (~2 min, no cluster): federated code still matches the NumPy reference -> ALL PASSED
!pip -q install pyspark==3.5.3
!python tests/test_federated_reference.py 2>/dev/null | grep -E "PASS|FAIL|ALL"

In [ ]:
# Plan check (no cluster): centralized + federated, strong + weak, 2 executors only
REPS = 1   # set to 3 for the final run
!python scripts/run_benchmarks.py --config config.yaml --modes centralized federated --executors 2 --repetitions {REPS} --skip-prepare --dry-run | grep -E "timed runs|=== run"

In [ ]:
# Only 2 executors are used, so a 2-worker cluster (3 machines instead of 17)
!CONFIG=config.yaml bash infra/dataproc_launch.sh 2

In [ ]:
!python scripts/run_benchmarks.py --config config.yaml --modes centralized federated --executors 2 --repetitions {REPS} --skip-prepare

In [ ]:
!CONFIG=config.yaml bash infra/dataproc_delete.sh